## Logistic Regression Sigmoid Log likelihood of Cross-Entropy Loss Optimization


In Logistic regression, we use the following linear function to compute the logit (z):  

$z = w^T x + b$

Where:
- $w$ is the weight vector.
- $x$ is the input feature vector.
- $b$ is the bias term.

And then we apply the sigmoid function to convert the logit into a probability (p):  

$p = \sigma(z) = \frac{1}{1 + e^{-z}}$

We want:  

$P(y=0|x) = 1$, if $y=0$,  
$P(y=1|x) = 1$, if $y=1$

Or in other terms:  

$P(y|x) = 
\begin{cases}
p & \text{if } y=1,\\[4pt]
1-p & \text{if } y=0
\end{cases}$

**Merge both cases using exponent trick:**  

Now, notice how exponents can “select” a term depending on whether $y$ is 0 or 1:

| $y$ | $p^y$ | $(1-p)^{1-y}$ | Product |
|---|-----|-------------|---------|
| 0 | $p^0=1$   | $(1-p)^1=1-p$         | $1-p$     |
| 1 | $p^1=p$   | $(1-p)^0=1$           | $p$       |


which can be written more compactly as:  

$P(y|x)=p^y(1−p)^{(1−y)}$.

**Example check**:

If $y=1$ and $p=0.9$:  
$P(y|x) = 0.9^1 \cdot (1-0.9)^{0} = 0.9$

If $y=0$ and $p=0.9$:  
$P(y|x) = 0.9^0 \cdot (1-0.9)^{1} = 0.1$

To achieve this, we maximize the likelihood over all training examples $1, …, n$:  

$P(y^{[i]},…,y^{[n]}|x^{[1]},…,x^{[n]})=∏_{i=1}^nP(y^{[i]}|x^{[i]})$

for which we can define the Likelihood function $L$:

$L(w)=P(y|x;w)$  
    $=∏_{i=1}^nP(y^{[i]}|x^{[i]};w)$  
    $=∏_{i=1}^n(σ(z^{[i]}))^{y^{[i]}}(1−σ(z^{[i]}))^{1−y^{[i]}}$

By taking the log, we arrive at an objective that is easier to optimize:  

$l(w)=logL(w)$  
$=∑_{i=1}^n[y^{[i]}log(σ(z^{[i]}))+(1−y^{[i]})log(1−σ(z^{[i]}))]$

Finally, to turn this maximization problem into a minimization problem that lets us use stochastic gradient descent optimizers in PyTorch, we are interested in the negative log likelihood:  

$\mathcal{L}(w)=-l(w)=-\sum_{i=1}^n[y^{(i)}\log(\sigma(z^{(i)}))+(1−y^{(i)})\log(1−\sigma(z^{(i)}))]$.

This is equivalent to the the binary cross entropy:

$\mathcal{H}_a(y)=-\sum_i(y^{[i]}log(a^{[i]})+(1−y^{[i]})log(1−a^{[i]}))$

And the generalized form of the cross entropy is extending this concept to $K$ classes (which are assumed to have a one-hot encoded form):

$\mathcal{H}_a(y)=\sum_{i=1}^n\sum_{k=1}^K - y^{[i]}_k\log(a^{[i]}_k)$

This is similar to the multinomial logistic loss, also known as softmax regression.

In short, cross-entropy is exactly the same as the negative log likelihood (these were two concepts that were originally developed independently in the field of computer science and statistics, and they are motivated differently, but it turns out that they compute excactly the same in our classification context.)

PyTorch mixes and matches these terms, which in theory are interchangeable. In PyTorch, these refer to implementations that accept different input arguments (but compute the same thing). This is summarized below.

### PyTorch Loss-Input Confusion (Cheatsheet)
- `torch.nn.functional.binary_cross_entropy` takes logistic sigmoid values as inputs
- `torch.nn.functional.binary_cross_entropy_with_logits` takes logits as inputs
- `torch.nn.functional.cross_entropy` takes logits as inputs (performs log_softmax internally)
- `torch.nn.functional.nll_loss` is like cross_entropy but takes log-probabilities (log-softmax) values as inputs

In [ ]:
# --- Numerically verify the identities derived above ---
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# Toy binary problem
z = torch.randn(5)                      # logits
y = torch.randint(0, 2, (5,)).float()   # binary labels
p = torch.sigmoid(z)                    # probabilities

# 1) The exponent "selection trick": P(y|x) = p^y * (1-p)^(1-y)
selected = p**y * (1 - p)**(1 - y)
piecewise = torch.where(y == 1, p, 1 - p)
print("selection trick == piecewise:", torch.allclose(selected, piecewise))

# 2) Negative log likelihood == binary cross entropy
nll = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
bce = F.binary_cross_entropy(p, y)                 # takes sigmoid probs
bce_logits = F.binary_cross_entropy_with_logits(z, y)  # takes raw logits
print(f"manual NLL = {nll.item():.6f}")
print("BCE(sigmoid)       == NLL:", torch.allclose(bce, nll))
print("BCE_with_logits(z) == NLL:", torch.allclose(bce_logits, nll))

# 3) Multiclass cheatsheet: cross_entropy(logits) == nll_loss(log_softmax)
Z = torch.randn(4, 3)                   # 4 samples, 3 classes
t = torch.randint(0, 3, (4,))
ce = F.cross_entropy(Z, t)              # logits in, log_softmax done internally
nll_ls = F.nll_loss(F.log_softmax(Z, dim=-1), t)   # log-probs in
print("cross_entropy == nll_loss(log_softmax):", torch.allclose(ce, nll_ls))